# Confidence change by accuracy transition and study arm

Summarize and visualize confidence change across accuracy trajectories and study arms.

## Configure analysis

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path("..").resolve()
DATA_PATH = ROOT / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"

VIGNETTES = ["A", "B", "C", "D", "E"]
ARM_LABELS = {1: "ChatGPT", 2: "Copilot", 3: "Perplexity", 4: "Control"}
ARM_ORDER = ["ChatGPT", "Copilot", "Perplexity", "Control"]
TRANSITION_ORDER = ["wrong→right", "right→right", "wrong→wrong", "right→wrong"]
OUTCOME_KEYS = ["condition", "triage"]

OUTCOMES = {
    "condition": {
        "label": "Condition diagnosis",
        "pre_suffix": "pre_condition_correct",
        "post_suffix": "post_condition_correct",
    },
    "triage": {
        "label": "Triage (next step)",
        "pre_suffix": "pre_nextstep_correct",
        "post_suffix": "post_nextstep_correct",
    },
}

## Load data

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

needed = ["prolific_id", "study_arm"]
for vignette in VIGNETTES:
    needed.extend(
        [
            f"vig_{vignette}_pre_confidence",
            f"vig_{vignette}_post_confidence",
            f"vig_{vignette}_pre_condition_correct",
            f"vig_{vignette}_post_condition_correct",
            f"vig_{vignette}_pre_nextstep_correct",
            f"vig_{vignette}_post_nextstep_correct",
        ]
    )

df = pd.read_excel(DATA_PATH)
missing = [col for col in needed if col not in df.columns]
if missing:
    raise KeyError(f"Columns not found: {missing}")

print(f"Loaded {len(df):,} participants")
print(df["study_arm"].value_counts().sort_index().rename(index=ARM_LABELS))

## Classify accuracy transitions

In [ ]:
def classify_transition(pre, post):
    if pd.isna(pre) or pd.isna(post):
        return pd.NA
    mapping = {
        (0, 1): "wrong→right",
        (1, 0): "right→wrong",
        (1, 1): "right→right",
        (0, 0): "wrong→wrong",
    }
    return mapping.get((int(pre), int(post)), pd.NA)


def build_long(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for vignette in VIGNETTES:
        pre_conf_col = f"vig_{vignette}_pre_confidence"
        post_conf_col = f"vig_{vignette}_post_confidence"

        for outcome_key, cfg in OUTCOMES.items():
            pre_correct_col = f"vig_{vignette}_{cfg['pre_suffix']}"
            post_correct_col = f"vig_{vignette}_{cfg['post_suffix']}"

            chunk = data[
                [
                    "prolific_id",
                    "study_arm",
                    pre_conf_col,
                    post_conf_col,
                    pre_correct_col,
                    post_correct_col,
                ]
            ].copy()
            chunk["vignette"] = vignette
            chunk["outcome"] = outcome_key
            chunk["transition"] = [
                classify_transition(pre, post)
                for pre, post in zip(chunk[pre_correct_col], chunk[post_correct_col])
            ]
            chunk = chunk.rename(
                columns={
                    pre_conf_col: "pre_confidence",
                    post_conf_col: "post_confidence",
                    pre_correct_col: "pre_correct",
                    post_correct_col: "post_correct",
                }
            )
            rows.append(chunk)

    long_df = pd.concat(rows, ignore_index=True)
    long_df["arm_label"] = pd.Categorical(
        long_df["study_arm"].map(ARM_LABELS), categories=ARM_ORDER, ordered=True
    )
    long_df["transition"] = pd.Categorical(
        long_df["transition"], categories=TRANSITION_ORDER, ordered=True
    )
    return long_df


long_df = build_long(df)
print(f"Long-format rows: {len(long_df):,} (805 participants × 5 vignettes × 2 outcomes)")
print("\nTransition counts — condition diagnosis (pooled vignettes):")
print(
    long_df.loc[long_df["outcome"] == "condition", "transition"]
    .value_counts()
    .reindex(TRANSITION_ORDER)
)

## Calculate confidence change

In [ ]:
long_df["delta_confidence"] = long_df["post_confidence"] - long_df["pre_confidence"]

analysis_df = long_df[
    long_df["transition"].notna()
    & long_df["delta_confidence"].notna()
    & long_df["arm_label"].notna()
].copy()

print(f"Analysis rows (complete transition + confidence): {len(analysis_df):,}")
print(
    analysis_df.groupby(["outcome", "arm_label"], observed=True)["delta_confidence"]
    .agg(n="count", mean="mean")
    .round(2)
)

## Summarize confidence change by trajectory and arm

Report observed means, 95% confidence intervals, and unadjusted one-sample tests within each cell.

In [ ]:
def mean_ci(values, alpha=0.05):
    series = pd.Series(values).dropna()
    n = len(series)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    mean = series.mean()
    if n == 1:
        return mean, np.nan, np.nan, n
    sem = series.std(ddof=1) / np.sqrt(n)
    tcrit = stats.t.ppf(1 - alpha / 2, n - 1)
    return mean, mean - tcrit * sem, mean + tcrit * sem, n


def one_sample_p(values) -> float:
    series = pd.Series(values).dropna()
    if len(series) < 2:
        return np.nan
    return float(stats.ttest_1samp(series, popmean=0.0).pvalue)


def format_p(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def summarize_by_arm_transition(data: pd.DataFrame, outcome_key: str) -> pd.DataFrame:
    sub = data[data["outcome"] == outcome_key]
    rows = []
    for arm in ARM_ORDER:
        for transition in TRANSITION_ORDER:
            cell = sub[(sub["arm_label"] == arm) & (sub["transition"] == transition)]
            mean, lo, hi, n = mean_ci(cell["delta_confidence"])
            p_value = one_sample_p(cell["delta_confidence"])
            rows.append(
                {
                    "outcome": OUTCOMES[outcome_key]["label"],
                    "arm": arm,
                    "transition": transition,
                    "n": n,
                    "mean_delta_confidence": round(mean, 2) if n else np.nan,
                    "ci_lower": round(lo, 2) if n > 1 else np.nan,
                    "ci_upper": round(hi, 2) if n > 1 else np.nan,
                    "p_value": p_value,
                    "p_display": format_p(p_value),
                    "significant_0.05": (p_value < 0.05) if pd.notna(p_value) else pd.NA,
                    "mean_ci_display": (
                        f"{mean:.1f} [{lo:.1f}, {hi:.1f}]" if n > 1 else (f"{mean:.1f}" if n == 1 else "")
                    ),
                }
            )
    return pd.DataFrame(rows)


summary_tables = {
    outcome_key: summarize_by_arm_transition(analysis_df, outcome_key)
    for outcome_key in OUTCOMES
}

for outcome_key, table in summary_tables.items():
    print(f"\n{'=' * 80}")
    print(OUTCOMES[outcome_key]["label"])
    print("=" * 80)
    display_cols = [
        "arm",
        "transition",
        "n",
        "mean_delta_confidence",
        "ci_lower",
        "ci_upper",
        "p_display",
        "significant_0.05",
    ]
    print(table[display_cols].to_string(index=False))

### Create compact summary tables

In [ ]:
def pivot_display(table: pd.DataFrame) -> pd.DataFrame:
    display = table.copy()
    display["cell"] = display.apply(
        lambda row: (
            f"{row['mean_delta_confidence']:.1f} [{row['ci_lower']:.1f}, {row['ci_upper']:.1f}], p={row['p_display']} (n={int(row['n'])})"
            if row["n"] > 1
            else (f"{row['mean_delta_confidence']:.1f} (n={int(row['n'])})" if row["n"] == 1 else "— (n=0)")
        ),
        axis=1,
    )
    return display.pivot(index="transition", columns="arm", values="cell").reindex(TRANSITION_ORDER)


def pivot_significance(table: pd.DataFrame) -> pd.DataFrame:
    sig = table.copy()
    sig["cell"] = sig["significant_0.05"].map({True: "Yes", False: "No"}).fillna("—")
    return sig.pivot(index="transition", columns="arm", values="cell").reindex(TRANSITION_ORDER)


for outcome_key, table in summary_tables.items():
    print(f"\n{OUTCOMES[outcome_key]['label']}")
    print(pivot_display(table).to_string())
    print("\nSignificant vs 0 (p < 0.05):")
    print(pivot_significance(table).to_string())

## Plot confidence change by accuracy trajectory

Show observed means with participant-clustered bootstrap confidence intervals and model-based LLM–control significance markers.

In [ ]:
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
from rpy2.robjects import default_converter, pandas2ri, r
from rpy2.robjects.conversion import localconverter

FIG_DIR = ROOT / "manuscript" / "Figures"
FIG_PNG = FIG_DIR / "Figure_confidence_change_by_transition.png"
FIG_PDF = FIG_DIR / "Figure_confidence_change_by_transition.pdf"

ARM_COLORS = {
    "ChatGPT": "#009E73",
    "Copilot": "#0072B2",
    "Perplexity": "#E69F00",
    "Control": "#999999",
}
CI_COLOR = "#888888"
N_BOOT = 2000
BOOT_RNG = np.random.default_rng(42)

TRANSITION_TO_R = {
    "wrong→right": "wrong_right",
    "right→right": "right_right",
    "wrong→wrong": "wrong_wrong",
    "right→wrong": "right_wrong",
}
TRANSITION_FROM_R = {v: k for k, v in TRANSITION_TO_R.items()}


def p_to_stars(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def cluster_bootstrap_summary(cell_df: pd.DataFrame, n_boot: int = N_BOOT) -> dict:
    """Cluster bootstrap CI on prolific_id; p from t-test on participant-level means."""
    cell_df = cell_df.dropna(subset=["delta_confidence", "prolific_id"])
    n = len(cell_df)
    if n == 0:
        return {
            "mean": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "n": 0,
            "p_value": np.nan,
            "significant_0.05": pd.NA,
        }

    obs_mean = float(cell_df["delta_confidence"].mean())
    by_participant = cell_df.groupby("prolific_id")["delta_confidence"].mean()

    if len(by_participant) < 2:
        return {
            "mean": obs_mean,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "n": n,
            "p_value": np.nan,
            "significant_0.05": pd.NA,
        }

    clusters = cell_df.groupby("prolific_id")["delta_confidence"].apply(lambda s: s.to_numpy()).to_dict()
    cluster_ids = np.array(list(clusters.keys()))
    boot_means = np.array(
        [
            np.concatenate(
                [clusters[cid] for cid in BOOT_RNG.choice(cluster_ids, len(cluster_ids), replace=True)]
            ).mean()
            for _ in range(n_boot)
        ]
    )
    p_value = float(stats.ttest_1samp(by_participant, popmean=0.0).pvalue)

    return {
        "mean": obs_mean,
        "ci_lower": float(np.percentile(boot_means, 2.5)),
        "ci_upper": float(np.percentile(boot_means, 97.5)),
        "n": n,
        "p_value": p_value,
        "significant_0.05": p_value < 0.05,
    }


def summarize_clustered_for_figure(data: pd.DataFrame, outcome_key: str) -> pd.DataFrame:
    sub = data[data["outcome"] == outcome_key]
    rows = []
    for transition in TRANSITION_ORDER:
        for arm in ARM_ORDER:
            cell = sub[(sub["arm_label"] == arm) & (sub["transition"] == transition)]
            rows.append(
                {
                    "outcome": OUTCOMES[outcome_key]["label"],
                    "outcome_key": outcome_key,
                    "transition": transition,
                    "arm": arm,
                    **cluster_bootstrap_summary(cell),
                }
            )
    out = pd.DataFrame(rows)
    out["p_display"] = out["p_value"].map(format_p)
    return out


def parse_contrast_df(ctr_df: pd.DataFrame) -> pd.DataFrame:
    p_col = "p.value" if "p.value" in ctr_df.columns else "p_value"
    rows = []
    for _, row in ctr_df.iterrows():
        arm_num = str(row["contrast"]).split(" - ")[0].replace("study_arm", "")
        rows.append(
            {
                "transition": TRANSITION_FROM_R[str(row["transition"])],
                "arm": ARM_LABELS[int(arm_num)],
                "p_value": float(row[p_col]),
            }
        )
    return pd.DataFrame(rows)


def lmer_contrasts_vs_control(data: pd.DataFrame, outcome_key: str) -> pd.DataFrame:
    sub = data[data["outcome"] == outcome_key].copy()
    sub["transition_r"] = sub["transition"].map(TRANSITION_TO_R)
    r_df = sub[
        ["prolific_id", "study_arm", "Age", "Sex", "vignette", "transition_r", "delta_confidence"]
    ].rename(columns={"transition_r": "transition"})
    r_df["study_arm"] = r_df["study_arm"].astype(int).astype(str)

    r("suppressPackageStartupMessages({library(lme4); library(emmeans)})")
    with localconverter(default_converter + pandas2ri.converter):
        r.assign("long_df", r_df)
    r("""
    long_df$study_arm <- relevel(factor(long_df$study_arm), ref = "4")
    long_df$transition <- factor(
      long_df$transition,
      levels = c("right_right", "wrong_right", "wrong_wrong", "right_wrong")
    )
    long_df$vignette <- factor(long_df$vignette, levels = c("A", "B", "C", "D", "E"))
    long_df$prolific_id <- factor(long_df$prolific_id)
    lmer_model <- lmer(
      delta_confidence ~ transition * study_arm + (1 | prolific_id) + (1 | vignette),
      data = long_df,
      REML = TRUE
    )
    emm <- emmeans(lmer_model, ~ study_arm | transition)
    ctr_df <- as.data.frame(summary(
      contrast(emm, "trt.vs.ctrl", adjust = "none"),
      infer = TRUE
    ))
    """)
    with localconverter(default_converter + pandas2ri.converter):
        ctr_df = r("ctr_df")
    out = parse_contrast_df(ctr_df)
    out["outcome_key"] = outcome_key
    return out


def star_arms_for_panel(contrast_panel: pd.DataFrame) -> dict[str, str]:
    stars = {}
    for row in contrast_panel.itertuples(index=False):
        label = p_to_stars(row.p_value)
        if label:
            stars[row.arm] = label
    return stars


def figure_y_limits(fig_df: pd.DataFrame, contrast_tables: dict[str, pd.DataFrame]) -> tuple[float, float]:
    valid = fig_df.dropna(subset=["ci_lower", "ci_upper"])
    if valid.empty:
        return -5.0, 25.0
    ymin = min(valid["ci_lower"].min(), 0.0)
    ymax = valid["ci_upper"].max()
    pad = max(4.0, 0.10 * (ymax - ymin))
    return ymin - pad, ymax + pad


def plot_transition_panel(
    ax,
    panel_df: pd.DataFrame,
    *,
    star_arms: dict[str, str],
    show_ylabel: bool = False,
) -> None:
    xs = np.arange(len(ARM_ORDER))
    max_top = 0.0
    transition = panel_df["transition"].iloc[0] if len(panel_df) else None
    for i, arm in enumerate(ARM_ORDER):
        row = panel_df[panel_df["arm"] == arm].iloc[0]
        color = ARM_COLORS[arm]
        if row["n"] == 0 or pd.isna(row["mean"]):
            continue

        sig = bool(row["significant_0.05"]) if pd.notna(row["significant_0.05"]) else False
        if arm == "Control" and transition == "right→wrong":
            facecolor = color
        else:
            facecolor = color if sig else "white"
        yerr = None
        if pd.notna(row["ci_lower"]) and pd.notna(row["ci_upper"]):
            yerr = np.array([[row["mean"] - row["ci_lower"]], [row["ci_upper"] - row["mean"]]])
            max_top = max(max_top, row["ci_upper"])

        ax.errorbar(
            i,
            row["mean"],
            yerr=yerr,
            fmt="o",
            markersize=6.5,
            markeredgewidth=1.4,
            markerfacecolor=facecolor,
            markeredgecolor=color,
            color=CI_COLOR,
            elinewidth=1.0,
            capsize=3,
            capthick=1.0,
            zorder=3,
        )
        if arm in star_arms and pd.notna(row["ci_upper"]):
            y_star = row["ci_upper"] + 1.8
            ax.text(
                i,
                y_star,
                star_arms[arm],
                ha="center",
                va="bottom",
                fontsize=9,
                color="#333333",
                zorder=5,
            )
            max_top = max(max_top, y_star + 2)

    ax.axhline(0, color="#666666", linewidth=0.9, linestyle="--", zorder=1)
    ax.set_xlim(-0.35, len(ARM_ORDER) - 0.65)
    ax.set_xticks(xs)
    ax.set_xticklabels(ARM_ORDER, rotation=0, ha="center")
    ax.tick_params(axis="x", pad=6)
    ax.set_ylabel("Δ confidence (post − pre)" if show_ylabel else "")
    ax.yaxis.grid(True, linestyle="-", alpha=0.25, color="#cccccc")
    ax.set_axisbelow(True)
    ax._panel_max_top = max_top  # noqa: SLF001


model_analysis_df = analysis_df.merge(df[["prolific_id", "Age", "Sex"]], on="prolific_id", how="left")
model_analysis_df = model_analysis_df.dropna(subset=["Age", "Sex"])

figure_tables = {
    outcome_key: summarize_clustered_for_figure(model_analysis_df, outcome_key)
    for outcome_key in OUTCOME_KEYS
}
contrast_tables = {
    outcome_key: lmer_contrasts_vs_control(model_analysis_df, outcome_key)
    for outcome_key in OUTCOME_KEYS
}

for outcome_key, table in figure_tables.items():
    print(f"\n{'=' * 88}")
    print(f"{OUTCOMES[outcome_key]['label']} — observed mean Δconfidence (cluster bootstrap CI)")
    print("=" * 88)
    print(
        table[
            ["transition", "arm", "n", "mean", "ci_lower", "ci_upper", "p_display", "significant_0.05"]
        ]
        .round(2)
        .to_string(index=False)
    )

for outcome_key, table in contrast_tables.items():
    print(f"\n{OUTCOMES[outcome_key]['label']} — LLM vs Control on Δconfidence (lmer)")
    print(table.assign(p_display=table["p_value"].map(format_p)).to_string(index=False))

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

fig, axes = plt.subplots(2, 4, figsize=(12, 7.2), sharey="row")
fig.subplots_adjust(wspace=0.28, hspace=0.58, top=0.86, bottom=0.16, left=0.09, right=0.98)

for row_idx, outcome_key in enumerate(OUTCOME_KEYS):
    fig_df = figure_tables[outcome_key]
    contrast_df = contrast_tables[outcome_key]
    ymin, ymax = figure_y_limits(fig_df, contrast_tables)

    for col_idx, transition in enumerate(TRANSITION_ORDER):
        ax = axes[row_idx, col_idx]
        panel_df = fig_df[fig_df["transition"] == transition]
        star_arms = star_arms_for_panel(contrast_df[contrast_df["transition"] == transition])
        plot_transition_panel(ax, panel_df, star_arms=star_arms, show_ylabel=(col_idx == 0))
        ax.set_title(transition, loc="left", fontsize=10.5, fontweight="bold", pad=8)
        panel_top = getattr(ax, "_panel_max_top", ymax)
        ax.set_ylim(ymin, max(ymax, panel_top + 2))

    row_y = 0.76 if row_idx == 0 else 0.31
    fig.text(
        0.03,
        row_y,
        OUTCOMES[outcome_key]["label"],
        ha="center",
        va="center",
        rotation=90,
        fontsize=11,
        fontweight="bold",
    )

arm_handles = [
    mlines.Line2D(
        [0], [0], marker="o", color="w", markerfacecolor=ARM_COLORS[arm],
        markeredgecolor=ARM_COLORS[arm], markersize=6.5, label=arm, linestyle="None",
    )
    for arm in ARM_ORDER
]
fig.legend(
    handles=arm_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.0),
)

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
print(f"\nSaved {FIG_PNG}")
print(f"Saved {FIG_PDF}")
plt.show()
